In [4]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(10)
random.seed(10)
n = 3000

# ---------- helper functions ----------
def random_ip():
    return ".".join(map(str, np.random.randint(0,255,4)))

def random_mac():
    return ":".join(["%02x" % random.randint(0,255) for _ in range(6)])

def random_time():
    start = datetime(2026,1,1)
    return start + timedelta(seconds=random.randint(0,500000))

# ---------- dataset ----------
data = {
    "Timestamp":[random_time() for _ in range(n)],
    "Source_IP":[random_ip() for _ in range(n)],
    "Destination_IP":[random_ip() for _ in range(n)],
    "Source_Port":np.random.randint(1024,65535,n),
    "Destination_Port":np.random.choice([21,22,53,80,443,8080],n),
    "Protocol":np.random.choice(["TCP","UDP","ICMP"],n),
    "Packet_Size":np.random.randint(40,2000,n),
    "Connection_Duration":np.random.randint(1,800,n),
    "Packets_Per_Second":np.random.randint(1,6000,n),
    "Error_Rate":np.random.random(n),
    "Login_Attempts":np.random.randint(0,25,n),
    "Bytes_Sent":np.random.randint(100,60000,n),
    "Bytes_Received":np.random.randint(100,70000,n),
    "Traffic_Flag":np.random.choice(["Normal","Suspicious","DoS","Probe"],n,p=[0.65,0.15,0.10,0.10]),  # realistic distribution
    "MAC_Address":[random_mac() for _ in range(n)]
}

df = pd.DataFrame(data)

# ---------- LABEL LOGIC ----------
# Label created BEFORE introducing missing values
df["Label"] = (
    ((df["Packet_Size"] > 1500) & (df["Error_Rate"] > 0.8)) |
    (df["Login_Attempts"] > 18) |
    ((df["Bytes_Sent"] > 50000) & (df["Traffic_Flag"] == "Suspicious")) |
    (df["Traffic_Flag"] == "DoS")
).astype(int)

# ---------- introduce realistic NULL values AFTER label ----------
df.loc[df.sample(frac=0.08).index,
       ["Packet_Size","Packets_Per_Second",
        "Bytes_Sent","Bytes_Received"]] = np.nan

df.loc[df.sample(frac=0.05).index,
       ["MAC_Address","Source_IP"]] = np.nan

df.loc[df.sample(frac=0.03).index, "Timestamp"] = np.nan

df.loc[df.sample(frac=0.06).index,
       ["Error_Rate","Connection_Duration"]] = np.nan

# ---------- save dataset as CSV File----------
df.to_csv("Network.csv", index=False)
print(df)
print(df["Label"].value_counts())

               Timestamp        Source_IP   Destination_IP  Source_Port  \
0    2026-01-04 11:12:59     9.125.228.15  115.144.135.238        13552   
1    2026-01-01 04:44:43   64.113.123.156    182.75.112.98        25373   
2    2026-01-03 14:27:41  217.221.240.157    225.58.187.19         3336   
3    2026-01-03 22:16:41     113.250.8.73   205.56.196.116         3114   
4    2026-01-04 12:11:26     0.234.40.246    47.149.140.64        56727   
...                  ...              ...              ...          ...   
2995 2026-01-01 04:51:20     185.58.52.13    175.152.56.75        26775   
2996 2026-01-03 10:06:36   219.133.118.49    29.78.154.210        63893   
2997 2026-01-03 04:13:06   115.131.210.60    251.99.186.40        54729   
2998 2026-01-03 19:41:45  101.150.166.139    101.170.214.5        50458   
2999 2026-01-01 08:38:01  103.122.151.184     89.29.65.108        15850   

      Destination_Port Protocol  Packet_Size  Connection_Duration  \
0                 8080      UD